# 05 · 量子神经网络：鸢尾花分类

对应讲义实验 D · 前置：01 编程基础（第 5 节）

**问题**：区分两种鸢尾花。每朵花有 4 个特征：花萼的长和宽、花瓣的长和宽。

**思路**：先用编码线路把特征写进量子态，再用可训练线路学习“测什么”：两个比特的期望值就是两类的得分。整个模型是一个普通的 MindSpore 网络层，用深度学习的方式训练；最后和经典模型比一比。

In [ ]:
import mqlab                         # 先导入：屏蔽 MindSpore 的冗长日志
import numpy as np
import matplotlib.pyplot as plt
import mindspore as ms
from mindspore.dataset import NumpySlicesDataset
from mindspore.nn import SoftmaxCrossEntropyWithLogits, Adam
from mindspore.train import Model, Accuracy
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from mindquantum import IQPEncoding, HardwareEfficientAnsatz, Ansatz1, X, RY
from mindquantum import Hamiltonian, QubitOperator, MQLayer, Simulator

## 1. 直觉：数据长什么样

鸢尾花数据集有 3 类、各 50 个样本。把花瓣的长和宽画出来：

In [ ]:
iris = load_iris()
plt.figure(figsize=(5, 3.2))
for c, name in enumerate(iris.target_names):
    pts = iris.data[iris.target == c]
    plt.scatter(pts[:, 2], pts[:, 3], s=14, label=f'{c}: {name}')
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.legend()
plt.show()

类别 0（山鸢尾）与另外两类一眼就能分开；类别 1 与 2 有重叠，难得多。先做简单的：区分类别 0 与 1。

## 2. 数据

取前两类，共 100 个样本。4 比特 IQP 编码线路有 7 个参数，对应 4 个特征本身，加上相邻特征的 3 个乘积（用于两比特相位项）。

In [ ]:
x = iris.data[:100].astype(np.float32)    # 类别 0 与 1，100 个样本
y = iris.target[:100].astype(np.int32)
x = np.hstack([x, x[:, :3] * x[:, 1:4]])  # 加 3 个乘积 → 7 维
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
print(x_train.shape, x_test.shape)

## 3. 模型：编码线路 + 可训练线路 + 测量

- `IQPEncoding(4)`：H 之后用单比特和两比特相位写入 7 个特征，参数不训练（就是 01 课后挑战中的线路）；
- `HardwareEfficientAnsatz`：RY + CNOT，共 3 层，16 个可训练参数；
- 测量 $\langle Z_2\rangle$、$\langle Z_3\rangle$，作为两类的得分（logits）。

In [ ]:
encoder = IQPEncoding(4).circuit                 # 7 个编码参数
ansatz = HardwareEfficientAnsatz(4, single_rot_gate_seq=[RY], entangle_gate=X, depth=3).circuit
circuit = encoder.as_encoder() + ansatz.as_ansatz()
print(len(encoder.params_name), '个编码参数，', len(ansatz.params_name), '个可训练参数')
circuit.svg()

## 4. 训练与评价

`MQLayer` 是标准的 MindSpore 网络层：损失函数、优化器、`Model.train` 都和经典深度学习完全一样。最后两行用逻辑回归作为经典基线。

In [ ]:
ms.set_seed(1)
hams = [Hamiltonian(QubitOperator(f'Z{i}')) for i in (2, 3)]
qnn = MQLayer(Simulator('mqvector', 4).get_expectation_with_grad(hams, circuit))  # 输出 ⟨Z2⟩、⟨Z3⟩

loss = SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
opt = Adam(qnn.trainable_params(), learning_rate=0.1)
model = Model(qnn, loss, opt, metrics={'acc': Accuracy()})
train_ds = NumpySlicesDataset({'features': x_train, 'labels': y_train}, shuffle=False).batch(5)
test_ds = NumpySlicesDataset({'features': x_test, 'labels': y_test}, shuffle=False).batch(5)
model.train(20, train_ds)                        # 20 轮

acc = model.eval(test_ds)['acc']
baseline = LogisticRegression(max_iter=1000).fit(x_train, y_train).score(x_test, y_test)
print(f'测试准确率：量子神经网络 {acc:.0%}，逻辑回归 {baseline:.0%}')

两者都是 100%：这两类线性可分，任务本身很简单。模型能学会，不等于比经典方法更好。

### 现场改一改：先预测，再运行

每次改一处，先写下预测，再从改动的单元格运行到第 4 节：

1. **换一个 Ansatz**：第 3 节改成 `ansatz = Ansatz1(4, 2).circuit`，同样 16 个参数，但没有纠缠门。测试准确率？　A 仍是 100%　B 降到 85%　C 降到 50%
2. **换随机种子**：把第 4 节的 `ms.set_seed(1)` 换成 2、3、7……，测试准确率稳定吗？　A 都是 100%　B 波动很大　C 约一半失败
3. **换一个更难的任务**：第 2 节改成 `x = iris.data[50:150].astype(np.float32)` 和 `y = (iris.target[50:150] - 1).astype(np.int32)`，即类别 1 与 2。量子模型与逻辑回归，谁更好？　A 量子模型　B 差不多　C 逻辑回归

<details><summary>答案</summary>

1. B：85%。参数一样多，结构不同，效果也不同。
2. A：都是 100%（种子 0–9 全部如此）：在这个简单任务上，训练很稳定。
3. C：量子模型 70%，逻辑回归 95%。简单任务上看不出差别，任务一变难，经典基线明显更好（对照第 1 节的散点图）。
</details>

## 课后练习（开放）

- 试试 `Ansatz2`，或把 `HardwareEfficientAnsatz` 的 depth 改为 1–5，比较测试准确率与参数个数；
- 在类别 1 与 2 上，量子模型最多能提高到多少？编码方式（去掉乘积特征、对特征做缩放）往往比 Ansatz 更关键。

## 小结

- 量子神经网络 = 编码线路 + 可训练线路 + 测量；在 MindSpore Quantum 中，它就是一个普通的网络层。
- 编码方式决定了数据如何进入量子态，往往比 Ansatz 更关键。
- 目前没有证据表明量子模型在实际数据上普遍优于经典模型：在系统的基准测试中，经典模型通常更好。把量子神经网络当作学习混合编程的练习，而不是性能承诺。